In [ ]:
from pathlib import Path
import os,sys,json,shutil,subprocess,importlib.util,hashlib,zipfile
from concurrent.futures import ThreadPoolExecutor,as_completed
import numpy as np,pandas as pd
from PIL import Image,ImageOps

FP='896491de87f9dc2a1d7d63548b7c5c22206da11f27a37efece8efc8e1557c8a9'
PTH=5; RTH=8; CORR=.98
W=Path('/kaggle/working'); O=W/'M07_NEAR_DUP_AUDIT'; O.mkdir(parents=True,exist_ok=True)
print('CGP_PHASE:M07_NEAR_DUP_SETUP')
for pkg,mod in [('kagglehub>=1.0.2','kagglehub'),('ImageHash','imagehash')]:
    if importlib.util.find_spec(mod) is None: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import kagglehub,imagehash

def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

def splitroot():
    roots=[Path('/kaggle/input'),W]
    for r in roots:
        if not r.exists(): continue
        for p in r.rglob('17_FROZEN_SPLIT_POLICY_AND_HASHES.json'):
            try:
                if json.loads(p.read_text())['split_package_fingerprint_sha256']==FP:return p.parent
            except: pass
    d=W/'_split'; shutil.rmtree(d,ignore_errors=True); d.mkdir()
    errs=[]
    for k in ['rezanory/m07-frozen-split-clone','radlinaradlina/m07-frozen-split-clone']:
        try:kagglehub.notebook_output_download(k,output_dir=str(d),force_download=True)
        except Exception as e:errs.append(repr(e))
        for p in d.rglob('17_FROZEN_SPLIT_POLICY_AND_HASHES.json'):
            try:
                if json.loads(p.read_text())['split_package_fingerprint_sha256']==FP:return p.parent
            except: pass
        for z in d.rglob('KERMANY_CLEAN_SPLIT_V1_FROZEN.zip'):
            x=W/'_splitx';shutil.rmtree(x,ignore_errors=True);x.mkdir()
            with zipfile.ZipFile(z) as q:
                for m in q.infolist():
                    t=(x/m.filename).resolve()
                    if not str(t).startswith(str(x.resolve())+os.sep):raise RuntimeError('UNSAFE_ZIP')
                q.extractall(x)
            for p in x.rglob('17_FROZEN_SPLIT_POLICY_AND_HASHES.json'):
                if json.loads(p.read_text())['split_package_fingerprint_sha256']==FP:return p.parent
    raise RuntimeError('FROZEN_SPLIT_NOT_FOUND '+str(errs))

S=splitroot(); pol=json.loads((S/'17_FROZEN_SPLIT_POLICY_AND_HASHES.json').read_text());assert pol['split_package_fingerprint_sha256']==FP
for n in ['11_locked_test_manifest.csv','14_development_with_fold_assignments.csv',*[f'fold_{i}_val.csv' for i in range(1,6)]]:
    p=S/n; assert p.is_file(),n
    if pol.get('manifest_hashes',{}).get(n):assert sha(p)==pol['manifest_hashes'][n],f'HASH:{n}'
print('FROZEN_SPLIT_VERIFIED',S)

def okroot(p):
    p=Path(p);return all((p/s/l).is_dir() for s in ['train','val','test'] for l in ['NORMAL','PNEUMONIA'])
def dataroot():
    for r in [Path('/kaggle/input'),W]:
        if r.exists():
            for p in r.rglob('chest_xray'):
                if okroot(p):return p
    d=W/'_raw';shutil.rmtree(d,ignore_errors=True);d.mkdir()
    try:g=Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia',output_dir=str(d),force_download=True))
    except Exception:
        subprocess.run(['kaggle','datasets','download','-d','paultimothymooney/chest-xray-pneumonia','-p',str(d),'--unzip','-o'],check=True);g=d
    for p in [g,g/'chest_xray',g/'chest_xray'/'chest_xray',*d.rglob('chest_xray')]:
        if okroot(p):return p
    raise RuntimeError('RAW_ROOT_NOT_FOUND')
D=dataroot();print('RAW_DATA_ROOT',D);print('CGP_PHASE:M07_NEAR_DUP_LOAD')

def load(n):
    a=pd.read_csv(S/n);need={'relative_path','patient_id','model_label','sha256'};assert need<=set(a.columns)
    a['filepath']=a.relative_path.astype(str).map(lambda x:str(D/x));assert a.filepath.map(lambda x:Path(x).is_file()).all();return a
dev=load('14_development_with_fold_assignments.csv');test=load('11_locked_test_manifest.csv')
fc=next((c for c in dev.columns if 'fold' in c.lower()),None)
if fc is None:
    m={}
    for i in range(1,6):
        for s in pd.read_csv(S/f'fold_{i}_val.csv').sha256.astype(str):m[s]=i
    fc='_audit_fold';dev[fc]=dev.sha256.astype(str).map(m);assert dev[fc].notna().all()
print('DEV',len(dev),dev.patient_id.nunique(),'TEST',len(test),test.patient_id.nunique(),'FOLD',fc)
pat=len(set(dev.patient_id.astype(str))&set(test.patient_id.astype(str))); sx=len(set(dev.sha256.astype(str))&set(test.sha256.astype(str)))
ddup=int(dev.duplicated('sha256',keep=False).sum());tdup=int(test.duplicated('sha256',keep=False).sum())
print('EXACT',pat,sx,ddup,tdup)

def hu(h):return np.uint64(int(str(h),16))
def hh(p):
    try:
        with Image.open(p) as im:
            g=ImageOps.exif_transpose(im).convert('L');return str(p),hu(imagehash.phash(g,hash_size=8,highfreq_factor=4)),hu(imagehash.dhash(g,hash_size=8)),None
    except Exception as e:return str(p),None,None,repr(e)
paths=pd.concat([dev.filepath,test.filepath]).astype(str).drop_duplicates().tolist();print('CGP_PHASE:M07_NEAR_DUP_HASH',len(paths))
r=[]
with ThreadPoolExecutor(max_workers=min(16,max(2,os.cpu_count() or 4))) as ex:
    fs=[ex.submit(hh,p) for p in paths]
    for i,f in enumerate(as_completed(fs),1):
        r.append(f.result())
        if i%500==0 or i==len(fs):print('HASHED',i,'/',len(fs))
h=pd.DataFrame(r,columns=['filepath','ph','dh','err']);bad=h[h.err.notna()]
if len(bad):bad.to_csv(O/'M07_HASH_ERRORS.csv',index=False);raise RuntimeError(f'HASH_ERRORS:{len(bad)}')
l=h.set_index('filepath')
for a in [dev,test]:a['ph']=a.filepath.map(l.ph);a['dh']=a.filepath.map(l.dh);assert a.ph.notna().all()
pc=np.array([int(i).bit_count() for i in range(256)],dtype=np.uint8)
def hm(a,b):
    x=np.bitwise_xor(np.asarray(a,np.uint64)[:,None],np.asarray(b,np.uint64)[None,:]);return pc[x.view(np.uint8).reshape(len(a),len(b),8)].sum(2,dtype=np.uint16)
def pairs(a,b,thr,name):
    a=a.reset_index(drop=True);b=b.reset_index(drop=True);rows=[];bh=b.ph.to_numpy(np.uint64)
    for s in range(0,len(a),96):
        z=hm(a.ph.iloc[s:s+96].to_numpy(np.uint64),bh);ii,jj=np.where(z<=thr)
        for i,j in zip(ii,jj):
            x=a.iloc[s+int(i)];y=b.iloc[int(j)];rows.append(dict(relation=name,phash_distance=int(z[i,j]),left_filepath=x.filepath,right_filepath=y.filepath,left_relative_path=x.relative_path,right_relative_path=y.relative_path,left_patient_id=x.patient_id,right_patient_id=y.patient_id,left_label=x.model_label,right_label=y.model_label,left_dhash=int(x.dh),right_dhash=int(y.dh)))
    return pd.DataFrame(rows)
def dd(a,b):return (int(a)^int(b)).bit_count()
def vc(a,b):
    def q(p):
        with Image.open(p) as im:im=ImageOps.fit(ImageOps.exif_transpose(im).convert('L'),(256,256),method=Image.Resampling.BILINEAR);x=np.asarray(im,np.float32)
        x-=x.mean();s=x.std();return x/s if s>1e-8 else x
    return float(np.mean(q(a)*q(b)))
print('CGP_PHASE:M07_NEAR_DUP_COMPARE')
dt=pairs(dev,test,RTH,'DEVELOPMENT_vs_LOCKED_TEST')
if len(dt):
    dt['dhash_distance']=[dd(a,b) for a,b in zip(dt.left_dhash,dt.right_dhash)];dt['visual_correlation']=[vc(a,b) for a,b in zip(dt.left_filepath,dt.right_filepath)]
    dt['published_phash_flag']=dt.phash_distance<=PTH;dt['very_strong_visual_match']=dt.published_phash_flag&(dt.dhash_distance<=5)&(dt.visual_correlation>=CORR);dt=dt.sort_values(['phash_distance','dhash_distance','visual_correlation'],ascending=[1,1,0])
else:dt=pd.DataFrame(columns=['phash_distance','dhash_distance','visual_correlation','published_phash_flag','very_strong_visual_match'])
cf=[];fv=sorted(dev[fc].dropna().unique())
for i in range(len(fv)):
    for j in range(i+1,len(fv)):
        x=pairs(dev[dev[fc]==fv[i]],dev[dev[fc]==fv[j]],PTH,f'FOLD_{fv[i]}_vs_{fv[j]}')
        if len(x):x['left_fold']=fv[i];x['right_fold']=fv[j];x['dhash_distance']=[dd(a,b) for a,b in zip(x.left_dhash,x.right_dhash)];cf.append(x)
cf=pd.concat(cf,ignore_index=True) if cf else pd.DataFrame()
dt.to_csv(O/'M07_NEAR_DUP_DEV_VS_TEST.csv',index=False);cf.to_csv(O/'M07_NEAR_DUP_CROSSFOLD.csv',index=False)
sumry={'split_fingerprint':FP,'development_images':int(len(dev)),'locked_test_images':int(len(test)),'development_patients':int(dev.patient_id.nunique()),'locked_test_patients':int(test.patient_id.nunique()),'patient_overlap_exact':pat,'sha256_overlap_exact':sx,'exact_duplicate_rows_inside_development':ddup,'exact_duplicate_rows_inside_test':tdup,'dev_test_phash_le_8':int(len(dt)),'dev_test_phash_le_5':int(dt.published_phash_flag.sum()) if len(dt) else 0,'dev_test_very_strong_matches':int(dt.very_strong_visual_match.sum()) if len(dt) else 0,'crossfold_phash_le_5':int(len(cf)),'published_phash_threshold':PTH}
(O/'M07_NEAR_DUP_AUDIT_SUMMARY.json').write_text(json.dumps(sumry,indent=2));print(json.dumps(sumry,indent=2))
verdict='PASS' if sumry['dev_test_phash_le_5']==0 and sumry['crossfold_phash_le_5']==0 else 'REVIEW_REQUIRED';print('M07_NEAR_DUP_AUDIT_'+verdict);print('CGP_PHASE:M07_NEAR_DUP_COMPLETE')
